[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import math

/home/codespace/.local/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [17]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    d_k = Q.size(-1)
    scores = torch.bmm(Q, torch.transpose(K, 1, 2)) / torch.sqrt(torch.tensor(d_k))
    seq = Q.size(1)
    future = torch.triu(torch.ones(seq, seq), diagonal=1).bool()
    scores.masked_fill_(future, -float('inf')) # since it goes to softmax
    res = torch.bmm(torch.softmax(scores, dim = -1), V)
    return res

In [18]:
torch.triu(torch.ones(4, 4), diagonal=1)

tensor([[0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.]])

Q shape = (batch, seq_q, d) = (1, 4, 8) → 4 queries, each an 8-d vector.
K shape = (batch, seq_k, d) = (1, 4, 8) → 4 keys, each an 8-d vector.
When you compute scores: scores = Q @ K.transpose(1,2) → (1, 4, 8) @ (1, 8, 4) = (1, 4, 4).
That final (1, 4, 4) is (batch, query_pos, key_pos): each query has 4 scores (one per key).
8 is the feature dimension (d), not the number of key positions or score columns. The score matrix has 4 columns (keys), not 8.

In [19]:
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)

Output shape: torch.Size([1, 4, 8])


(batch, seq_k, d_v)

In [20]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [21]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (163.1ms)
  ✅ [2/4] Future tokens don't affect past (6.9ms)
  ✅ [3/4] First position only sees itself (1.1ms)
  ✅ [4/4] Gradient flow (68.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (239.8ms total)
  Progress saved. Run status() to see your dashboard.

